In [27]:
import numpy as np
import trimesh
from dlinvc.agents import CoarseGeometryAgent, SemanticLabel, CoarseScene
from scipy.spatial.transform import Rotation
import pyrender
from PIL import Image
import h5py

In [2]:
agent = CoarseGeometryAgent()

with open("../prompts/scene_generator_prompt.txt", "r") as f:
    prompt = f.read()

scene = agent.generate_geometry("../hyperism/ai_001_001/images/scene_cam_00_final_preview/frame.0000.color.jpg", prompt)

In [6]:
scene.objects

[ObjectBoundingBox(id='floor_01', label=<SemanticLabel.FLOOR: 'floor'>, extents=[4.0, 4.0, 0.1], center=[2.0, 2.0, 0.05], orientation=[0.0, 0.0, 0.0, 1.0]),
 ObjectBoundingBox(id='ceiling_01', label=<SemanticLabel.CEILING: 'ceiling'>, extents=[4.0, 4.0, 0.1], center=[2.0, 2.0, 3.05], orientation=[0.0, 0.0, 0.0, 1.0]),
 ObjectBoundingBox(id='wall_left_01', label=<SemanticLabel.WALL: 'wall'>, extents=[0.1, 4.0, 3.0], center=[0.05, 2.0, 1.5], orientation=[0.0, 0.0, 0.0, 1.0]),
 ObjectBoundingBox(id='wall_back_01', label=<SemanticLabel.WALL: 'wall'>, extents=[4.0, 0.1, 3.0], center=[2.0, 4.05, 1.5], orientation=[0.0, 0.0, 0.0, 1.0]),
 ObjectBoundingBox(id='wall_right_01', label=<SemanticLabel.WALL: 'wall'>, extents=[0.1, 4.0, 3.0], center=[4.05, 2.0, 1.5], orientation=[0.0, 0.0, 0.0, 1.0]),
 ObjectBoundingBox(id='bathtub_01', label=<SemanticLabel.FURNITURE: 'furniture'>, extents=[1.8, 0.8, 0.6], center=[0.95, 1.2, 0.3], orientation=[0.0, 0.0, 0.0, 1.0]),
 ObjectBoundingBox(id='shelf_01', l

In [12]:
boxes = []
for obj in scene.objects:
    if obj.label == SemanticLabel.WALL:
        continue

    mesh = trimesh.creation.box(extents=np.array(obj.extents))
    R = Rotation.from_quat(obj.orientation).as_matrix()

    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = obj.center

    mesh.apply_transform(T)
    boxes.append(mesh)

tm_scene = trimesh.util.concatenate(boxes)

In [17]:
c2w = np.array(
    [
        [7.41193831e-01, -3.51180434e-01, 5.72105408e-01, 1.03228874e02],
        [5.79402149e-01, -9.57124457e-02, -8.09399724e-01, -1.17227142e02],
        [3.39002311e-01, 9.31402802e-01, 1.32532686e-01, 6.23017311e01],
        [0.00000000e00, 0.00000000e00, 0.00000000e00, 1.00000000e00],
    ]
)

fov_y = np.float64(0.8172757347349127)

aspect_ratio = 4 / 3

width_pixels = 1024
height_pixels = 768

In [20]:
pr_mesh = pyrender.Mesh.from_trimesh(tm_scene)
pr_scene = pyrender.Scene()
pr_scene.add(pr_mesh)

camera = pyrender.PerspectiveCamera(yfov=fov_y, aspectRatio=width_pixels / height_pixels)
pr_scene.add(camera, pose=c2w)

light = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=3.0)
pr_scene.add(light, pose=c2w)

r = pyrender.OffscreenRenderer(width_pixels, height_pixels)
color, depth = r.render(pr_scene)

img = Image.fromarray(color)
img.save("../outputs/rendered_agentic.png")

In [47]:
def extract_center_points(scene: CoarseScene):
    centers = []
    for obj in scene.objects:
        if obj.label == SemanticLabel.WALL or obj.label == SemanticLabel.FLOOR or obj.label == SemanticLabel.CEILING:
            continue
        centers.append(obj.center)
    return np.array(centers)


def align_icp(source_points: np.ndarray, target_points: np.ndarray):
    M_a_to_b, T, cost = trimesh.registration.icp(
        source_points, target_points, max_iterations=1000, initial=np.eye(4), threshold=1e-5
    )
    return M_a_to_b, T, cost

In [48]:
centers = extract_center_points(scene)
centers.shape

(12, 3)

In [28]:
with h5py.File(
    "../hyperism/ai_001_001/_detail/mesh/metadata_semantic_instance_bounding_box_object_aligned_2d_positions.hdf5"
) as f:
    positions = np.array(f["dataset"])

In [35]:
positions = positions[~np.isnan(positions).any(axis=1)]
positions = positions[~np.isinf(positions).any(axis=1)]

positions.shape

(55, 3)

In [57]:
M_a_to_b, T, cost = align_icp(centers, positions)
R = M_a_to_b[:3, :3]
t = M_a_to_b[:3, 3]
R_rot = Rotation.from_matrix(R)

In [58]:
boxes = []
for bbox in scene.objects:
    if obj.label == SemanticLabel.WALL or obj.label == SemanticLabel.FLOOR or obj.label == SemanticLabel.CEILING:
        continue

    new_center = (R @ np.array(bbox.center)) + t

    R_obj = R_rot * Rotation.from_quat(bbox.orientation)
    T_obj = np.eye(4)
    T_obj[:3, :3] = R_obj.as_matrix()
    T_obj[:3, 3] = new_center

    mesh = trimesh.creation.box(extents=np.array(obj.extents), transform=T_obj)
    boxes.append(mesh)

tm_scene = trimesh.util.concatenate(boxes)